# Phase 1 — The Dataset
## Brain Tumour MRI Classification
====================================================================

Acquire, audit, clean, split, and fix the preprocessing constants. Nothing is
trained here.

The requirement this notebook carries is that `Training/` and `Testing/` share
no images. On this dataset that is not the default, and section 1 explains why
the shipped split cannot simply be taken at face value.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src import audit, config, data, manifest, splits, viz
from src.config import CACHE_DIR, CLASSES, TEST_DIR, TRAIN_DIR

In [2]:
# 1. WHY THE SHIPPED SPLIT CANNOT BE TAKEN AT FACE VALUE
"""
This dataset arrives pre-split into Training/ and Testing/, which invites you to
use it as-is. Doing that produces a number that is wrong, and wrong in a way no
ordinary evaluation reveals, because the flaw is in the split rather than in the
model.

Three defects, all measured rather than assumed, and all reproduced by this
notebook further down:

Repeated scans across the split. Hundreds of test images are pixel-identical to
training images under different filenames. A model scored on those is being
tested on its memory, not its generalisation.

Repeated scans inside Training/ itself. A split that assigns images
independently puts one copy of a scan in train and another in validation, so
the number every training decision is made against is inflated too.

Pre-augmented copies. The meningioma class was padded by the dataset author with
transformed duplicates -- 100 in Training, 103 in Testing, all with '-aug-' in
the filename. Removing them from Testing alone is the obvious half and not
enough, since the copies and their originals both sit in Training.

There is a fourth problem this notebook can measure but not fix, and section 4
reports it: the classes come from different source collections with different
native resolutions, so image size predicts the label. That one is a property of
the data.

The response to all of this is not to write a caveat. It is to remove the
leakage and then assert that it is gone -- section 10 -- and to report the
shortcut beside every headline number rather than in a footnote.
"""
print(f"dataset  {config.KAGGLE_DATASET}")
print(f"root     {config.TRAIN_DIR.parent}")
print(f"classes  {CLASSES}")

dataset  masoudnickparvar/brain-tumor-mri-dataset
root     C:\Users\AMAY M NAIR\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\2
classes  ['glioma', 'meningioma', 'notumor', 'pituitary']


In [3]:
# 2. THE DATASET AS SHIPPED
"""
Counted before anything is removed, so the removals in later sections can be
checked against a known starting point.
"""
print(f"{'split':<10}{'class':<14}{'files':>8}{'-aug- copies':>15}")
print("-" * 47)
shipped = {}
for split_dir in (TRAIN_DIR, TEST_DIR):
    total = 0
    for cls_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        files = list(cls_dir.iterdir())
        aug = [f for f in files if '-aug-' in f.name.lower()]
        total += len(files)
        print(f"{split_dir.name:<10}{cls_dir.name:<14}{len(files):>8}{len(aug):>15}")
    shipped[split_dir.name] = total
print("-" * 47)
for k, v in shipped.items():
    print(f"{k:<10}{'TOTAL':<14}{v:>8}")

split     class            files   -aug- copies
-----------------------------------------------
Training  glioma            1400              0
Training  meningioma        1400            100
Training  notumor           1400              0
Training  pituitary         1400              0
Testing   glioma             400              0
Testing   meningioma         400            103
Testing   notumor            400              0
Testing   pituitary          400              0
-----------------------------------------------
Training  TOTAL             5600
Testing   TOTAL             1600


In [4]:
# 3. DECODING ONCE, AND DROPPING THE AUTHOR'S COPIES
"""
Reading every JPEG on every access is the dominant cost here: num_workers must
stay 0 on Windows and the ablation re-reads the training set dozens of times.
Decoding produces the same pixels every epoch, so it happens once, into a uint8
array held in RAM and memoised on disk. Augmentation still runs per access,
because that is the part that is supposed to differ every epoch.

The cache is stored at 224px rather than the 128px we train at, so resolution
can be ablated later without rebuilding it. Paths are cached alongside the
pixels: evaluation stratifies accuracy by native file size, which is impossible
if the cache forgets which file each row came from.

The '-aug-' copies are dropped here, from both splits.
"""
train_img, train_lab, train_paths, test_img, test_lab, test_paths, C = \
    splits.build_caches(TRAIN_DIR, TEST_DIR)
assert C == CLASSES, "class order on disk does not match config"

print(f"train cache {train_img.shape}  {train_img.nbytes/1e6:.0f} MB")
print(f"test  cache {test_img.shape}  {test_img.nbytes/1e6:.0f} MB")
print(f"\n5600 shipped - 100 augmented = {len(train_lab)} train")
print(f"1600 shipped - 103 augmented = {len(test_lab)} test")
print(f"\nexample path retained: {Path(train_paths[0]).name}")

train cache (5500, 224, 224)  276 MB
test  cache (1497, 224, 224)  75 MB

5600 shipped - 100 augmented = 5500 train
1600 shipped - 103 augmented = 1497 test

example path retained: Tr-gl_1.jpg


In [5]:
# 4. THE AUDIT GATE
"""
One question, asked before a single weight is trained: can anything other than
the anatomy predict the label?

Four checks. The decisive one is the last: a random forest given only width,
height, file size and bytes-per-pixel, which never sees a pixel. If it beats
chance then the labels are partly recoverable from the files themselves, and
whatever the CNN reports is partly a measurement of that.

This runs first because the alternative was tried. In an earlier version of this
project a model was trained, evaluated and written up before anyone discovered
it was reading a source signature rather than anatomy.

Two of these checks are expected to fail. That is the point -- the failure is
recorded now rather than discovered after the write-up.
"""
AUDIT = audit.run_audit(TRAIN_DIR, TEST_DIR, CLASSES,
                        train_cache=train_img, test_cache=test_img,
                        train_lab=train_lab, test_lab=test_lab)

DATASET AUDIT  --  6997 images after dropping '-aug-' copies
A1  native resolution by class
    distinct (w,h) across both splits: 447
    glioma        1724/1800   at 512x512 ( 95.8%)   62 distinct
    meningioma    1347/1597   at 512x512 ( 84.3%)   126 distinct
    notumor         20/1800   at 512x512 (  1.1%)   250 distinct
    pituitary     1720/1800   at 512x512 ( 95.6%)   27 distinct
    -> spread in 512x512 share across classes: 94.7%

A2  JPEG quantization tables and bytes-per-pixel by class
    class         distinct tables   top share   mean bpp
    glioma                      2      77.7%     0.0806
    meningioma                  3      56.0%     0.1037
    notumor                    50      70.0%     0.1944
    pituitary                   2      51.7%     0.1150
    -> tables shared by all classes: 2

A3  repeated scans


    redundant copies inside Training/   464 of 5500
    redundant copies inside Testing/    74 of 1497
    test images repeating a train scan  293 of 1497  (19.6%)
      glioma          7 / 400
      meningioma    102 / 297
      notumor       184 / 400
      pituitary       0 / 400

A4  metadata-only probe (no pixels)


    5-fold accuracy 0.6158 +/- 0.0361   chance 0.2500


    importance: bpp 0.345  filesize 0.335  width 0.190  height 0.130
    -> lift over chance: +0.3658

VERDICT
  FAIL  metadata probe near chance
  FAIL  no class separable by size
  PASS  quantization tables shared
  FAIL  no cross-split repeats

  -> Something other than anatomy can predict the label here.
     Duplicates are removed downstream and re-checked. The size
     shortcut cannot be removed, so evaluation reports every
     headline number beside its size-stratified breakdown.


In [6]:
# 5. WHAT THE AUDIT FOUND, AND WHAT HAPPENS TO EACH FINDING
"""
The two failures are different in kind and get different treatment.

The duplicates are removable, and sections 8 to 10 remove them and then prove
they are gone. Nothing about the shortcut is removable: the tumour classes come
overwhelmingly from 512x512 source images and notumor almost never does, so
image size carries real information about the label. Resizing everything to
128px hides the dimensions but not the resampling signature, and an augmentation
built specifically to destroy that signature was measured, on this dataset, to
change nothing. Augmentation cannot invent training examples of a class in a
style that occurs zero times under that label.

So the shortcut is carried forward as a reporting obligation rather than a bug:
evaluation stratifies every headline number by native file size, and the
stratified figure is quoted beside the aggregate wherever the aggregate appears.
"""
for label in AUDIT["failed"]:
    print(f"  FAILED  {label}")
print()
print(f"metadata-only accuracy  {AUDIT['probe']['accuracy']:.4f}  "
      f"(chance {AUDIT['probe']['chance']:.4f}, lift {AUDIT['probe']['lift']:+.4f})")
print(f"512x512 share spread across classes  {AUDIT['resolution']['spread_512']:.1%}")
print(f"cross-split repeats  {AUDIT['duplicates']['cross_split']} of {len(test_lab)}")
print(f"repeats inside Training/  {AUDIT['duplicates']['train_redundant']}")

  FAILED  metadata probe near chance
  FAILED  no class separable by size
  FAILED  no cross-split repeats

metadata-only accuracy  0.6158  (chance 0.2500, lift +0.3658)
512x512 share spread across classes  94.7%
cross-split repeats  293 of 1497
repeats inside Training/  464


In [7]:
# 6. CROPPING TO CONTENT
"""
A third to a half of a raw slice is empty background, and it is not a consistent
fraction -- these scans come from different fields of view, so the brain occupies
a different share of each image. Resizing without cropping therefore rescales
the brain by an arbitrary per-image factor, and the network has to spend
capacity ignoring a nuisance variable that can simply be deleted.

It has a second use: RandomAffine fills the corners it rotates into with a
constant, and fill=0 is only the right choice if everything outside the brain
really is black.

The figure and the measurement below are how that is checked rather than
assumed.
"""
sample_path = sorted((TRAIN_DIR / CLASSES[0]).iterdir())[0]
raw = Image.open(sample_path).convert("L")
cropped = data.crop_to_content(raw)

fig, axes = viz.styled_fig(1, 3, figsize=(11, 4))
axes[0].imshow(raw, cmap='gray');     axes[0].set_title(f"raw {raw.size}", fontsize=9)
axes[1].imshow(cropped, cmap='gray'); axes[1].set_title(f"cropped {cropped.size}", fontsize=9)
axes[2].imshow(cropped.resize((128, 128)), cmap='gray')
axes[2].set_title("cropped + resized 128", fontsize=9)
for ax in axes: ax.axis('off')
plt.suptitle("Cropping removes a per-image scale difference", fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "preprocessing.png")

kept = []
for p in train_paths[::50]:
    im = Image.open(p).convert("L")
    kept.append(np.prod(data.crop_to_content(im).size) / np.prod(im.size))
print(f"\narea kept after cropping, over {len(kept)} sampled scans:")
print(f"  mean {np.mean(kept):.1%}   min {np.min(kept):.1%}   max {np.max(kept):.1%}")
print("-> cropping removes real background; the step earns its place.")

  saved -> outputs/preprocessing.png

area kept after cropping, over 110 sampled scans:
  mean 81.8%   min 41.8%   max 100.0%
-> cropping removes real background; the step earns its place.


In [8]:
# 7. THE SAME SCAN ON BOTH SIDES, LOOKED AT
"""
Section 4 counted the repeats. This looks at them, because a claim that two
images are the same scan is worth checking with your eyes once.

Three bands, and the middle and lower ones are the interesting part. Pixel-
identical pairs are obvious. What matters is that just below the duplicate
threshold sit pairs which are visibly the same patient one slice apart --
identical ventricles and skull outline, shifted a few millimetres. No pixel test
calls those duplicates, and without patient identifiers nothing else can either.

The statistic that settles it: above cosine 0.95 every single pair shares a
class, against roughly 77% for unrelated nearest neighbours. Coincidental
lookalikes do not agree on the label 100% of the time.
"""
_dup, _match, _cos = data.find_duplicates(test_img, train_img)
_l1 = data.contrast_l1(test_img, train_img[_match])

print(f"{'cosine band':<20}{'n':>6}{'same class':>13}{'median L1':>12}")
print("-" * 51)
for lo, hi, lab in ((0.999, 1.01, '>= 0.999'), (0.995, 0.999, '0.995 - 0.999'),
                    (0.98, 0.995, '0.98  - 0.995'), (0.95, 0.98, '0.95  - 0.98'),
                    (0.90, 0.95, '0.90  - 0.95'), (0.0, 0.90, '< 0.90')):
    m = (_cos >= lo) & (_cos < hi)
    if not m.sum():
        continue
    same = (train_lab[_match[m]] == test_lab[m]).mean()
    print(f"{lab:<20}{m.sum():>6}{same:>12.1%}{np.median(_l1[m]):>12.3f}")

bands = []
for lo, hi, title in ((0.9995, 1.01, "IDENTICAL  -  pixel-for-pixel the same file"),
                      (0.98, 0.995, "NEAR  0.98-0.995  -  same patient, adjacent slice"),
                      (0.95, 0.98, "SIMILAR  0.95-0.98  -  same patient, further apart")):
    idx = np.where((_cos >= lo) & (_cos < hi))[0]
    idx = idx[np.argsort(-_cos[idx])]
    seen, pairs = set(), []
    for i in idx:
        if _match[i] in seen:
            continue
        seen.add(_match[i])
        pairs.append((test_img[i], f"TEST  {CLASSES[test_lab[i]]}\n{Path(test_paths[i]).stem}",
                      train_img[_match[i]],
                      f"TRAIN  {CLASSES[train_lab[_match[i]]]}  {Path(train_paths[_match[i]]).stem}",
                      f"cos={_cos[i]:.4f}   pixel L1={_l1[i]:.3f}"))
        if len(pairs) == 5:
            break
    if pairs:
        bands.append((title, pairs))
viz.plot_pairs(bands)

cosine band              n   same class   median L1
---------------------------------------------------
>= 0.999               266       99.2%       0.015
0.995 - 0.999           27      100.0%       0.058
0.98  - 0.995           64      100.0%       0.121
0.95  - 0.98            85      100.0%       0.219
0.90  - 0.95           265       99.2%       0.282
< 0.90                 790       77.3%       0.428


  saved -> outputs/leakage_examples.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/leakage_examples.png')

In [9]:
# 8. REMOVING THE LEAKAGE
"""
Three removals, in order, all performed by src/splits.py rather than here.
Logic that decides what the model is measured against does not belong in a
notebook cell -- in an earlier version it did, and a stale editor tab wrote an
older copy of that notebook back to disk, silently reverting the split to a
leaky one while the run that followed looked entirely normal.

  1. redundant copies inside Training/, keeping one image per cluster. These
     do not leak across the split, but an image present twice silently counts
     twice in the loss.
  2. test images matching anything still in Training/. The match is against the
     surviving training images, not the original set, so the exclusion describes
     the data the model will actually see.
  3. repeats inside Testing/ itself. Also not cross-split leakage, but a scan
     scored twice carries double weight in every metric.

Clustering happens at cosine 0.95 rather than the duplicate threshold of 0.995,
so that near-duplicates are kept together too. Connected components stay tight
because the pixel-L1 condition still has to hold -- measured, the largest
cluster is nine images even at 0.92.

Which files were dropped and why is written to outputs/excluded.json, because a
count in a notebook is not auditable and a list of filenames is.
"""
S = splits.build_splits(train_img, train_lab, test_img)
train_idx, val_idx = S["train_idx"], S["val_idx"]
train_keep, test_leak = S["train_keep"], S["test_leak"]
KEEP = ~test_leak

print(f"TRAIN  {len(train_lab)} -> {train_keep.sum()} after dropping "
      f"{(~train_keep).sum()} redundant -> {len(train_idx)} train + {len(val_idx)} val")
print(f"TEST   {len(test_lab)} -> {KEEP.sum()} after dropping {test_leak.sum()}")

E = splits.write_exclusions(train_paths, test_paths, S)
print(f"\noutputs/excluded.json  {E['counts']}")
print(f"rule: {E['rule']}")

TRAIN  5500 -> 4928 after dropping 572 redundant -> 4224 train + 704 val
TEST   1497 -> 1027 after dropping 470

outputs/excluded.json  {'train_dropped_as_redundant': 572, 'test_dropped_as_leaked': 470}
rule: {'duplicate_cosine': 0.995, 'duplicate_pixel_l1': 0.15, 'strict_leak': True, 'strict_leak_cosine': 0.95, 'dedupe_train': True}


In [10]:
# 9. THE SPLIT
"""
Testing/ is set aside and not looked at again until evaluation. Training/ is
split into train and validation, stratified so each class keeps its proportions
and grouped so no near-duplicate cluster crosses the boundary.

The validation set is what every later decision is made against -- which epoch
to checkpoint, when to stop, which configuration wins. That is precisely why it
cannot also be the set we report: a number chosen as the maximum of many is
biased upward, and the test set exists to give an estimate nothing was selected
on.

Grouping costs exact control over the split size, because whole clusters only
divide so finely. That is a fair price for a validation set that is genuinely
held out.
"""
print(f"train {len(train_idx):>6}   val {len(val_idx):>5}   test {KEEP.sum():>5}\n")
print(f"{'class':<14}{'train':>8}{'val':>7}{'test':>7}")
print("-" * 36)
for c, name in enumerate(CLASSES):
    print(f"{name:<14}{(train_lab[train_idx]==c).sum():>8}"
          f"{(train_lab[val_idx]==c).sum():>7}{((test_lab==c)&KEEP).sum():>7}")
print(f"\nsplit hash {S['split_hash']}")

train   4224   val   704   test  1027

class            train    val   test
------------------------------------
glioma            1200    200    375
meningioma        1099    184    190
notumor            764    127     88
pituitary         1161    193    374

split hash 053deb1b6524f6ca


In [11]:
# 10. UNIQUENESS, VERIFIED
"""
The requirement, checked rather than asserted. Six relationships have to be
empty: repeats inside each of the three splits, and repeats across each pair of
them.

assert_no_leakage deliberately recomputes the comparison from scratch instead of
reusing the mask that produced the exclusion. A check that trusts the thing it
is checking verifies nothing.

One residual is reported and not removed. The val-to-train figure sits above the
0.95 clustering threshold because those pairs clear the cosine bar but fail the
pixel-L1 test, so they are not duplicates by the rule and both conditions have
to hold. It matters less than it looks -- validation drives selection, not the
reported number -- but it is stated rather than rounded away.
"""
TR, VA, TE = train_img[train_idx], train_img[val_idx], test_img[KEEP]

total = 0
print("WITHIN each split")
for cache, name in ((TR, "train"), (VA, "val"), (TE, "test")):
    g = data.duplicate_groups(cache)
    extra = len(g) - len(np.unique(g))
    total += extra
    print(f"  {name:<8}{len(cache):>6} images{extra:>5} redundant")

print("\nACROSS splits")
for a, b, na, nb in ((VA, TR, "val ", "train"), (TE, TR, "test", "train"),
                     (TE, VA, "test", "val  ")):
    d, _, cos = data.find_duplicates(a, b)
    total += int(d.sum())
    print(f"  {na} -> {nb:<8}{d.sum():>5} duplicates   max cosine {cos.max():.4f}")

gate = splits.assert_no_leakage(train_img[train_keep], test_img, KEEP)
print(f"\n  independent re-check: 0 duplicates across {gate['checked']} test images,"
      f" max cosine {gate['max_cosine']:.4f}")
assert total == 0, f"{total} duplicate relationships remain"
print("\n  ALL IMAGES UNIQUE")

WITHIN each split


  train     4224 images    0 redundant
  val        704 images    0 redundant


  test      1027 images    0 redundant

ACROSS splits


  val  -> train       0 duplicates   max cosine 0.9915


  test -> train       0 duplicates   max cosine 0.9499


  test -> val         0 duplicates   max cosine 0.9474



  independent re-check: 0 duplicates across 1027 test images, max cosine 0.9499

  ALL IMAGES UNIQUE


In [12]:
# 11. NORMALISATION FROM THE TRAINING SPLIT ONLY
"""
Normalisation shifts and scales every pixel by two constants learned from data,
which makes them exactly as capable of leaking as model weights: computing them
over the whole dataset lets information about the held-out scans reach the
training pipeline.

They are computed after cropping and resizing, so they describe the tensors the
model actually receives rather than some earlier version of them.
"""
MEAN, STD = data.compute_stats(train_img, train_idx)
np.save(CACHE_DIR / "norm.npy", np.array([MEAN, STD]))
print(f"MEAN {MEAN:.4f}   STD {STD:.4f}")
print(f"computed over the {len(train_idx)} training images only "
      f"-- not validation, not test")

MEAN 0.2200   STD 0.1840
computed over the 4224 training images only -- not validation, not test


In [13]:
# 12. AUGMENTATION
"""
Which transforms are defensible on brain scans is a separate question from
whether augmentation helps, and it is the one a medical imaging report gets
challenged on.

No RandomHorizontalFlip. The usual argument against it -- that brains are not
symmetric -- is weak, because axial and coronal slices are grossly bilaterally
symmetric and tumours occur on both sides. The real problem is that this dataset
mixes acquisition planes, and a mirrored sagittal slice is anatomically
impossible.

RandomAffine stays: small rotations, shifts and scalings correspond to real
variation in how a patient was positioned in the scanner.

ColorJitter stays, mildly. MRI intensity is not calibrated in absolute physical
units the way CT Hounsfield numbers are -- scanner, sequence and windowing all
shift brightness and contrast -- so jittering reproduces genuine acquisition
variance rather than inventing a distortion that never happens.

Blur and noise are available and off by default. They address a different
objection: a model trained on uniformly clean images learns to depend on that,
and previously collapsed from 0.977 to 0.579 under moderate noise. Whether they
belong in the final recipe is a training decision, measured in a later phase.
"""
train_tf = data.make_transforms(MEAN, STD, augment=True)
eval_tf  = data.make_transforms(MEAN, STD, augment=False)
print("train:", *[f"  {t.__class__.__name__}" for t in train_tf.transforms], sep="\n")
print("\neval: ", *[f"  {t.__class__.__name__}" for t in eval_tf.transforms], sep="\n")

src_img = Image.fromarray(train_img[train_idx[0]], mode="L")
fig, axes = viz.styled_fig(1, 6, figsize=(13, 2.6))
axes[0].imshow(src_img, cmap='gray'); axes[0].set_title("original", fontsize=8)
for ax in axes[1:]:
    ax.imshow(viz.denorm(train_tf(src_img), MEAN, STD), cmap='gray')
    ax.set_title("augmented", fontsize=8)
for ax in axes: ax.axis('off')
plt.suptitle("One scan, five augmented views", fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "augmentation.png")

train_ds = data.CachedDataset(train_img, train_lab, train_tf, train_idx)
viz.show_batch(train_ds, CLASSES, MEAN, STD, n=8,
               title="Training batch after the full pipeline")

train:
  Resize
  RandomAffine
  ColorJitter
  ToTensor
  Normalize

eval: 
  Resize
  ToTensor
  Normalize


  saved -> outputs/augmentation.png
  saved -> outputs/sample_batch.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/sample_batch.png')

In [14]:
# 13. CLASS BALANCE AFTER CLEANING
"""
The classes are uneven, and the reason matters for the write-up: this is a
consequence of cleaning honestly, not a property of the disease. notumor lost
the most because its source collection spread each subject's slices across both
shipped folders, so removing same-patient bleed removed more of that class than
of any other.

Training is mildly uneven. The test set is not -- notumor is down to a small
fraction of the largest class, which is why evaluation reports macro-averaged
metrics rather than plain accuracy, and why the test set is not balanced by
discarding data. Truncating every class to the smallest would throw away roughly
two thirds of the held-out images to make one number look tidy.

The correction itself is a training decision and is applied in a later phase.
Three options are implemented -- inverse-frequency loss weights, a balanced
sampler, and undersampling -- and the choice between them is measured rather
than assumed. What is fixed here is only the measurement.
"""
counts = {name: [int((lab == c).sum()) for c in range(len(CLASSES))]
          for name, lab in (("train", train_lab[train_idx]),
                            ("val", train_lab[val_idx]),
                            ("test", test_lab[KEEP]))}
print(f"{'split':<8}" + "".join(f"{c:>13}" for c in CLASSES) + f"{'imbalance':>12}")
print("-" * (8 + 13 * len(CLASSES) + 12))
for name, c in counts.items():
    print(f"{name:<8}" + "".join(f"{v:>13}" for v in c) + f"{max(c)/min(c):>11.2f}:1")

w = data.class_weights(train_lab[train_idx], len(CLASSES))
print(f"\ninverse-frequency loss weights (option '{config.BALANCE}'):")
for n, v in zip(CLASSES, w.tolist()):
    print(f"  {n:<14}{v:.4f}")

viz.plot_class_balance(counts["train"], CLASSES,
                       title="Class balance in the training split")

split          glioma   meningioma      notumor    pituitary   imbalance
------------------------------------------------------------------------
train            1200         1099          764         1161       1.57:1
val               200          184          127          193       1.57:1
test              375          190           88          374       4.26:1

inverse-frequency loss weights (option 'weights'):
  glioma        0.8800
  meningioma    0.9609
  notumor       1.3822
  pituitary     0.9096
  saved -> outputs/class_balance.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/class_balance.png')

In [15]:
# 14. SUMMARY AND VERIFICATION
"""
Everything downstream depends on the objects fixed here: the caches, the split
indices, the exclusion record, the class list read from disk, and the
normalisation constants. The checks below fail loudly and name what failed --
a bare assertion that everything passed tells you nothing at two in the morning.

The run manifest is written last. It records the dataset digest, the split
digest, the configuration and the git commit, and its short hash is stamped on
every figure this project produces.
"""
DATASET_HASH = splits._hash(train_lab, test_lab)
RUN_HASH = manifest.write(DATASET_HASH, S["split_hash"], extra={
    "audit_failed": AUDIT["failed"],
    "metadata_probe_lift": round(AUDIT["probe"]["lift"], 4),
    "train_dropped": int((~train_keep).sum()),
    "test_dropped": int(test_leak.sum()),
    "n_train": len(train_idx), "n_val": len(val_idx), "n_test": int(KEEP.sum()),
})

checks = [
    ("class order on disk matches config",     C == CLASSES),
    ("augmented copies dropped from both",     len(train_lab) == 5500 and len(test_lab) == 1497),
    ("train and val indices disjoint",         not (set(train_idx) & set(val_idx))),
    ("no cluster straddles train/val",         not (set(S['groups'][train_idx]) &
                                                    set(S['groups'][val_idx]))),
    ("zero duplicates anywhere",               total == 0),
    ("exclusion record written",               (config.OUTPUTS / "excluded.json").exists()),
    ("normalisation from train split only",    0.0 < MEAN < 1.0 and STD > 0),
    ("audit ran and recorded a verdict",       "failed" in AUDIT),
    ("run manifest written",                   config.MANIFEST_PATH.exists()),
]
print("=" * 68)
print("PHASE 1 VERIFICATION")
print("=" * 68)
for label, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
failed = [label for label, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  cache        train {train_img.shape}, test {test_img.shape}
  split        {len(train_idx)} train / {len(val_idx)} val / {KEEP.sum()} test
  removed      {(~train_keep).sum()} redundant training, {test_leak.sum()} leaked test
  uniqueness   0 duplicate relationships within or across any split
  audit        {len(AUDIT['failed'])} check(s) failed: {', '.join(AUDIT['failed']) or 'none'}
  normalise    MEAN {MEAN:.4f}  STD {STD:.4f}
  run hash     {RUN_HASH}

  The dataset is fixed. Carried forward: the caches, norm.npy and splits.npz on
  disk, plus outputs/excluded.json as the auditable record of what was dropped.
  Nothing here has touched Testing/ beyond counting, auditing and cleaning it.""")

PHASE 1 VERIFICATION
  OK    class order on disk matches config
  OK    augmented copies dropped from both
  OK    train and val indices disjoint
  OK    no cluster straddles train/val
  OK    zero duplicates anywhere
  OK    exclusion record written
  OK    normalisation from train split only
  OK    audit ran and recorded a verdict
  OK    run manifest written

  cache        train (5500, 224, 224), test (1497, 224, 224)
  split        4224 train / 704 val / 1027 test
  removed      572 redundant training, 470 leaked test
  uniqueness   0 duplicate relationships within or across any split
  audit        3 check(s) failed: metadata probe near chance, no class separable by size, no cross-split repeats
  normalise    MEAN 0.2200  STD 0.1840
  run hash     7b3c5e5f9a85

  The dataset is fixed. Carried forward: the caches, norm.npy and splits.npz on
  disk, plus outputs/excluded.json as the auditable record of what was dropped.
  Nothing here has touched Testing/ beyond counting, auditing